# 03 — Matching and balance

**Goal.** Build a matched comparison sample for each pool, and prove it is actually balanced.

| | |
|---|---|
| **Reads** | `outputs/data/02_scored_cps.csv`, `02_scored_psid.csv` |
| **Writes** | `outputs/data/03_pairs_cps.csv`, `03_pairs_psid.csv` |

**Questions this notebook answers**

1. What caliper, and on which scale, raw probability or logit?
2. With or without replacement, and what does that choice do to the estimand?
3. **How many treated units survived matching?** If a large share is dropped, the quantity
   being estimated is no longer the ATT for the full treated population, and nothing in the
   standard output will warn you.
4. Is every |SMD| below 0.1 afterwards?

`psm.nn_match` returns one row per pair with an explicit `pair_id`, so the pairing stays
auditable instead of depending on row order.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import warnings

warnings.filterwarnings("ignore", message=".*numexpr.*")
warnings.filterwarnings("ignore", message=".*bottleneck.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import psm, data, figures

pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

POOLS = ["cps", "psid"]

%matplotlib inline

In [ ]:
scored = {pool: data.load_stage(f"02_scored_{pool}") for pool in POOLS}

for pool, df in scored.items():
    ps = df["prop_score"]
    print(f"{pool:5s} n={len(df):>6,}   prop_score  treated mean {ps[df.treat==1].mean():.3f}"
          f"   control mean {ps[df.treat==0].mean():.3f}")